# Kickstart project using python codeblocks
**IMPORTANT:** Keep in mind that you must load the python environment kernel with all dependencies installed and have the database running before hand.

This is a notebook to quickly show the flow of importation of the dataset from csv to a relational database (postgres) using python within the notebook.

## Pre-requirements
First start the necessary imports from `src/` and external libraries and start the required variables for this notebook to work:

In [21]:
%load_ext autoreload
%autoreload 2

# Enhanced console output
from src.config.logging import start_logger

# File handling utilities
from src.utils.files import read_file

# SQL utilities
from src.utils.sql import (
    connect_to_db,
    execute_sql_commands,
    copy_expert
)

# Paths
from src.paths import (
    SQL_DIR,
    DATA_DIR,
    PROJECT_ROOT
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
log = start_logger(show_path=False)  # Configure logging with custom settings

## Init database tables
We run a function for initializing the creation of tables proposed in phase 1 of the project. The `init.sql` script creates the relational tables allowing Null values (data cleanse not done yet).

`read_file` returns a `str` of the stream of the file; `execute_sql_commands` runs the `str` as a command using a connector to the database.

In [23]:
def init_database():
    log.info("Initializing database...")
    
    # init.sql file path
    init_sql_file_path = SQL_DIR / "init.sql"
    sql_commands = read_file(init_sql_file_path)

    execute_sql_commands(connect_to_db(), sql_commands)
    log.info("Database initialized successfully.")

init_database()

[22:48:16] INFO     Initializing database...

[22:48:16] DEBUG    File found at 'src\sql\init.sql'

[22:48:16] DEBUG    File at 'src\sql\init.sql' read successfully with 353 characters

╭─────────────────────────────────────────────── Commands Executed ───────────────────────────────────────────────╮
│    1 CREATE TABLE IF NOT EXISTS clientes (                                                                      │
│    2     id int PRIMARY KEY,                                                                                    │
│    3     edad float,                                                                                            │
│    4     genero text,                                                                                           │
│    5     fecha_registro date                                                                                    │
│    6 );                                                                                                         │
│    7                                                                                                            │
│    8 CREATE TABLE IF NOT EXISTS visitas_mensuales (                                                             │
│    9     cliente_id int REFERENCES clientes(id),                                                                │
│   10     cuenta_visitas int                                                                                     │
│   11 );                                                                                                         │
│   12                                                                                                            │
│   13 CREATE TABLE IF NOT EXISTS compra_restaurante (                                                            │
│   14     cliente_id int REFERENCES clientes(id),                                                                │
│   15     metodo_pago text,                                                                                      │
│   16     monto_compra float                                                                                     │
│   17 );                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[22:48:16] INFO     SQL commands executed successfully

[22:48:16] INFO     Database initialized successfully.

## Import dataset from csv to relational tables
This process consists on :
1. Creating a staging table with all columns set as `text` and allowing null values (`temp_stage_table.sql`)
2. Using `COPY` command to copy the stream of the dataset as csv. This is achieved through `psycopg2.copy_expert()`, which recieves a valid copy command as `str` (stored at `inser_command.sql`) and the payload to be copied as an object (`src.sql.copy_expert()` simplifies this process)
3. Insert all imports using the stage table to fill the data (`insert_command.sql`)

In [24]:
def import_dataset():
    log.info("Importing dataset script...")
    conn = connect_to_db()
    
    # Paths to SQL scripts and CSV file
    temp_stage_table_sql_path = SQL_DIR / "temp_stage_table.sql"
    insert_imports_sql_path = SQL_DIR / "insert_imports.sql"
    copy_command_sql_path = SQL_DIR / "copy_command.sql"
    csv_file_path = DATA_DIR / "dataset_clientes_cafeteria.csv"

    # Execute 'temp_stage_table.sql' to create the temporary staging table
    execute_sql_commands(conn, read_file(temp_stage_table_sql_path))

    # Execute 'copy_command.sql' to load data from the CSV file into the staging table
    copy_expert(conn, read_file(copy_command_sql_path), csv_file_path)

    # Execute 'insert_imports.sql' to transfer data from the staging table to the main tables
    execute_sql_commands(conn, read_file(insert_imports_sql_path))

    log.info("Dataset imported successfully.")
    conn.close()

import_dataset()

[22:48:16] INFO     Importing dataset script...

[22:48:16] DEBUG    File found at 'src\sql\temp_stage_table.sql'

[22:48:16] DEBUG    File at 'src\sql\temp_stage_table.sql' read successfully with 187 characters

╭─────────────────────────────────────────────── Commands Executed ───────────────────────────────────────────────╮
│   1 CREATE TEMP TABLE stage_clientes (                                                                          │
│   2     cliente_id text,                                                                                        │
│   3     edad text,                                                                                              │
│   4     genero text,                                                                                            │
│   5     monto_compra text,                                                                                      │
│   6     metodo_pago text,                                                                                       │
│   7     visitas_mensuales text,                                                                                 │
│   8     fecha_registro text                                                                                     │
│   9 );                                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[22:48:16] INFO     SQL commands executed successfully

...

[22:48:16] DEBUG    File found at 'src\sql\insert_imports.sql'

[22:48:16] DEBUG    File at 'src\sql\insert_imports.sql' read successfully with 540 characters

╭─────────────────────────────────────────────── Commands Executed ───────────────────────────────────────────────╮
│    1 INSERT INTO clientes (id, edad, genero, fecha_registro)                                                    │
│    2 SELECT                                                                                                     │
│    3     cliente_id::int,                                                                                       │
│    4     NULLIF(edad, '')::float,                                                                               │
│    5     NULLIF(genero, '')::text,                                                                              │
│    6     NULLIF(fecha_registro, '')::date                                                                       │
│    7 FROM stage_clientes;                                                                                       │
│    8                                                                                                            │
│    9 INSERT INTO visitas_mensuales (cliente_id, cuenta_visitas)                                                 │
│   10 SELECT                                                                                                     │
│   11     cliente_id::int,                                                                                       │
│   12     NULLIF(visitas_mensuales, '')::int                                                                     │
│   13 FROM stage_clientes;                                                                                       │
│   14                                                                                                            │
│   15 INSERT INTO compra_restaurante (cliente_id, metodo_pago, monto_compra)                                     │
│   16 SELECT                                                                                                     │
│   17     cliente_id::int,                                                                                       │
│   18     NULLIF(metodo_pago, '')::text,                                                                         │
│   19     NULLIF(monto_compra, '')::float                                                                        │
│   20 FROM stage_clientes;                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[22:48:17] INFO     SQL commands executed successfully

[22:48:17] INFO     Dataset imported successfully.

## Export back to csv
As the last part of this phase, the `export_database()` writes csv files for each table within the `tables` list.

In [25]:
def export_database():
    tables = ["clientes", "visitas_mensuales", "compra_restaurante"]

    for table in tables:
        log.info(f"Exporting table '{table}' to CSV...")
        file_path = DATA_DIR / "export" / f"{table}.csv"
        copy_expert(connect_to_db(), f"COPY {table} TO STDOUT WITH CSV HEADER", file_path)
        log.info(f"Table '{table}' exported successfully to '{file_path.relative_to(PROJECT_ROOT)}'")

    log.info("All tables exported successfully.")

export_database()

[22:48:17] INFO     Exporting table 'clientes' to CSV...

╭───────────────────────────────────────────── COPY Command Executed ─────────────────────────────────────────────╮
│   1 COPY clientes TO STDOUT WITH CSV HEADER                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[22:48:17] INFO     Data exported successfully to 'src\data\export\clientes.csv'

[22:48:17] INFO     Table 'clientes' exported successfully to 'src\data\export\clientes.csv'

[22:48:17] INFO     Exporting table 'visitas_mensuales' to CSV...

...

[22:48:17] INFO     Exporting table 'compra_restaurante' to CSV...

╭───────────────────────────────────────────── COPY Command Executed ─────────────────────────────────────────────╮
│   1 COPY compra_restaurante TO STDOUT WITH CSV HEADER                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[22:48:17] INFO     Data exported successfully to 'src\data\export\compra_restaurante.csv'

[22:48:17] INFO     Table 'compra_restaurante' exported successfully to 'src\data\export\compra_restaurante.csv'

[22:48:17] INFO     All tables exported successfully.